In [1]:
import pandas as pd
import numpy as np

In [2]:
df_debit = pd.read_parquet("data/ready_to_train/df_debit_final_25_feats_20250702.parquet")

In [3]:
all_feats = [col for col in df_debit.columns if col not in ["Transaction Datetime","Confirmed"]]
category_feats = ["POSMode","HighRiskCustomer"]
numerical_feats = [col for col in all_feats if col not in category_feats]

In [4]:
for col in category_feats:
    if col == "POSMode":
        df_debit[col] = df_debit[col].apply(lambda x: np.nan if isinstance(x, str) and x.strip() == '' else x)
        df_debit[col] = df_debit[col].fillna("__missing__")
        df_debit[col] = df_debit[col].astype(str).str.strip()
    df_debit[col] = df_debit[col].fillna("__missing__")

for col in numerical_feats:
    df_debit[col] = df_debit[col].fillna(0)

In [5]:
df_train_template = pd.read_csv("data/others/0_Training Template (Debit).csv")
df_validation_template = pd.read_csv("data/others/0_Validation Template (Debit).csv")

In [6]:
rename_col_map = {
    'Transaction Serial No': 'Identifier',
    'Confirmed': 'Fraud_Clean',
    'AvgTimeFirstTxnToCurrentMCCL30D': 'Transaction_Summary_Calculations_Fraud.AvgTimeFirstTxnToCurrentMCCL30D.1',
    'TransactionAmount': 'Transaction_Summary_Fraud.Transaction_Amount.1',
    'TotalTrxAmount10Mi': 'Transaction_Summary_Calculations_Fraud.TotalTrxAmount10Mi.1',
    'RatioTxnCountL30DL15min': 'Transaction_Summary_Calculations_Fraud.RatioTxnCountL30DL15min.1',
    'TxnTimeDifference': 'Transaction_Summary_Calculations_Fraud.TxnTimeDifference.1',
    'HighRiskCustomer': 'Transaction_Summary_Calculations_Fraud.HIghRiskCustomer.1',
    'POSMode': 'Transaction_Summary_Calculations_Fraud.POSMode_Std.1',
    'CustomerAge': 'Transaction_Summary_Calculations_Fraud.CustomerAge.1',
    'MaxAmtL30D': 'Transaction_Summary_Calculations_Fraud.MaxAmtL30D.1',
    'AvgAmtL30D': 'Transaction_Summary_Calculations_Fraud.AvgAmtL30D.1',
    'SumAmtL30D': 'Transaction_Summary_Calculations_Fraud.SumAmtL30D.1',
    'TotalTrxAmountL5min': 'Transaction_Summary_Calculations_Fraud.TotalTrxAmountL5min.1',
    'TotalTrxAmount15Mi': 'Transaction_Summary_Calculations_Fraud.TotalTrxAmount15Mi.1',
    'TotalTrxAmountL1D': 'Transaction_Summary_Calculations_Fraud.TotalTrxAmountL1D.1',
    'RatioTrxAmountL1DL15min': 'Transaction_Summary_Calculations_Fraud.RatioTrxAmountL1DL15min.1',
    'RatioTrxAmountL1DL10min': 'Transaction_Summary_Calculations_Fraud.RatioTrxAmountL1DL10min.1',
    'RatioTrxAmountL1DL5min': 'Transaction_Summary_Calculations_Fraud.RatioTrxAmountL1DL5min.1',
    'CntUniqueCardNoByMCCL15min': 'Transaction_Summary_Calculations_Fraud.CntUniqueCardNoByMCCL15min.1',
    'CntUniqueCardNoByMCCL30D': 'Transaction_Summary_Calculations_Fraud.CntUniqueCardNoByMCCL30D.1',
    'RatioCntUniqueCardNoByMCCL30DL15min': 'Transaction_Summary_Calculations_Fraud.RatioCntUniqueCardNoByMCCL30DL15min.1',
    'AvgTrnxHourL15min': 'Transaction_Summary_Calculations_Fraud.AvgTxnHourL15min.1',
    'TxnCountL30D': 'Transaction_Summary_Calculations_Fraud.TxnCountL30D.1',
    'AvgTrnxHourL30d': 'Transaction_Summary_Calculations_Fraud.AvgTxnHourL30D.1',
    'TimeFirstTxnToCurrentMCC': 'Transaction_Summary_Calculations_Fraud.DurationFirstTxnToCurrentMCC.1',
    'CountTrxTrf': 'Transaction_Summary_Calculations_Fraud.CountTrxTrf.1'
}

In [7]:
df_debit_final = df_debit.reset_index().rename(columns=rename_col_map)

In [8]:
df_debit_final['Fraud_Clean'] = df_debit_final['Fraud_Clean'].astype(int)

In [9]:
from src.model_pipeline import ModelPipeline

pipeline = ModelPipeline()

split_date = '2025-05-01'
df_splits = pipeline.split_data_by_date(
    df=df_debit_final,
    date_column="Transaction Datetime",
    split_date=split_date
)

Initializing ModelPipeline with model type: xgboost
Retrieving model instance for type: xgboost
Splitting data by date (pre-preprocessing)...
Train samples: 647760, Test samples: 66974
Data splitting complete.


In [10]:
df_splits["df_train"] = df_splits["df_train"].drop("Transaction Datetime", axis=1)
df_splits["df_train"][df_train_template.columns.to_list()].to_csv("data/ready_to_train/TRAIN_DEBIT_df_sit__int_label_20250710.csv", index=False)

df_splits["df_test"] = df_splits["df_test"].drop("Transaction Datetime", axis=1)
df_splits["df_test"][df_validation_template.columns.to_list()].to_csv("data/ready_to_train/VALIDATION_DEBIT_df_sit__int_label_20250710.csv", index=False)